# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


In [1]:
#%idle_timeout 3000
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

# Setting up the Spark and Glue contexts
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Session ID: cb1c0950-201f-421a-99c2-1f858c631305
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
Waiting for session cb1c0950-201f-421a-99c2-1f858c631305 to get into ready status...
Session cb1c0950-201f-421a-99c2-1f858c631305 has been created.



In [4]:
### Casting the data types to upload into data ware house for campaign reports

pc = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/ConstantContact/pearmundcellars-cc-email-campaigns-report-2024-10-01-2024-10-31.csv")
ec = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/ConstantContact/effinghammanor-cc-email-campaigns-report-2024-10-01-2024-10-31.csv")

pc = pc.dropDuplicates()
ec = ec.dropDuplicates()

from pyspark.sql import functions as F

m_pc = pc \
    .withColumn("Time Sent", F.date_format(F.to_timestamp(F.col("Time Sent"), "yyyy/MM/dd h:mm a"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("Sends", F.col("Sends").cast("int")) \
    .withColumn("Opens", F.col("Opens").cast("int")) \
    .withColumn("Open Rate", (F.regexp_replace("Open Rate", "%", "").cast("float"))) \
    .withColumn("Mobile Open Rate", (F.regexp_replace("Mobile Open Rate", "%", "").cast("float"))) \
    .withColumn("Desktop Open Rate", (F.regexp_replace("Desktop Open Rate", "%", "").cast("float"))) \
    .withColumn("Clicks", F.col("Clicks").cast("int")) \
    .withColumn("Click Rate", (F.regexp_replace("Click Rate", "%", "").cast("float"))) \
    .withColumn("Bounces", F.col("Bounces").cast("int")) \
    .withColumn("Bounce Rate", (F.regexp_replace("Bounce Rate", "%", "").cast("float"))) \
    .withColumn("Unsubscribes", F.col("Unsubscribes").cast("int")) \
    .withColumn("Unsubscribe Rate", (F.regexp_replace("Unsubscribe Rate", "%", "").cast("float")))

m_ec = ec \
    .withColumn("Time Sent", F.date_format(F.to_timestamp(F.col("Time Sent"), "yyyy/MM/dd h:mm a"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("Sends", F.col("Sends").cast("int")) \
    .withColumn("Opens", F.col("Opens").cast("int")) \
    .withColumn("Open Rate", (F.regexp_replace("Open Rate", "%", "").cast("float"))) \
    .withColumn("Mobile Open Rate", (F.regexp_replace("Mobile Open Rate", "%", "").cast("float"))) \
    .withColumn("Desktop Open Rate", (F.regexp_replace("Desktop Open Rate", "%", "").cast("float"))) \
    .withColumn("Clicks", F.col("Clicks").cast("int")) \
    .withColumn("Click Rate", (F.regexp_replace("Click Rate", "%", "").cast("float"))) \
    .withColumn("Bounces", F.col("Bounces").cast("int")) \
    .withColumn("Bounce Rate", (F.regexp_replace("Bounce Rate", "%", "").cast("float"))) \
    .withColumn("Unsubscribes", F.col("Unsubscribes").cast("int")) \
    .withColumn("Unsubscribe Rate", (F.regexp_replace("Unsubscribe Rate", "%", "").cast("float")))


In [5]:
# Loading the files to S3 after casting
s3_path_m_pc = "s3://pearmundeffinghamwinery/Output/ConstantContact/pc_oct"
s3_path_m_ec = "s3://pearmundeffinghamwinery/Output/ConstantContact/em_oct"


m_pc.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_m_pc)
m_ec.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_m_ec)

In [4]:
# Casting the tock transactions data
pt = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/Tock/pearmundcellars-24145-Tock-Transactions-2024-10-01-2024-10-31.csv")
et = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/Tock/effinghammanor-24146-Tock-Transactions-2024-10-01-2024-10-31.csv")

pt = pt.dropDuplicates()
et = et.dropDuplicates()

from pyspark.sql import functions as F
m_pt = pt.select(
    F.col("Action").alias("Action"),
    F.col("First Name").alias("First_Name"),
    F.col("Last Name").alias("Last_Name"),
    F.col("Email").alias("Email"),
    F.col("Phone").alias("Phone"),
    F.col("Party Size").cast("int").alias("Party_Size"),
    F.col("Experience").alias("Experience"),
    F.col("Fees").alias("Fees"),
    F.col("Price Per Person").cast("decimal(10, 2)").alias("Price_per_Person"),
    F.to_date(F.col("Booking Date"), "yyyy-MM-dd").alias("Booking_date"),
    # Convert Booking Time from 12-hour to 24-hour format
    F.date_format(F.to_timestamp(F.col("Booking Time"), "h:mm a"), "HH:mm:ss").alias("Booking_time"), # Cast to TIME
    F.col("Total Price").cast("decimal(10, 2)").alias("Total_Price")
)
m_et = et.select(
    F.col("Action").alias("Action"),
    F.col("First Name").alias("First_Name"),
    F.col("Last Name").alias("Last_Name"),
    F.col("Email").alias("Email"),
    F.col("Phone").alias("Phone"),
    F.col("Party Size").cast("int").alias("Party_Size"),
    F.col("Experience").alias("Experience"),
    F.col("Fees").alias("Fees"),
    F.col("Price Per Person").cast("decimal(10, 2)").alias("Price_per_Person"),
    F.to_date(F.col("Booking Date"), "yyyy-MM-dd").alias("Booking_date"),
    # Convert Booking Time from 12-hour to 24-hour format
    F.date_format(F.to_timestamp(F.col("Booking Time"), "h:mm a"), "HH:mm:ss").alias("Booking_time"), # Cast to TIME
    F.col("Total Price").cast("decimal(10, 2)").alias("Total_Price")  
)

In [5]:
# Loading the files to S3 after casting
s3_path_m_pt = "s3://pearmundeffinghamwinery/Output/Tock/pc_oct"
s3_path_m_et = "s3://pearmundeffinghamwinery/Output/Tock/em_oct"

m_pt.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_m_pt)
m_et.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_m_et)

In [8]:
# Casting the data types of OrderPort report

po = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/OrderPort/pearmundcellars-op-Ordered-Products-2024-10-01-2024-10-31.csv")
eo = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/OrderPort/effinghammanor-op-Ordered-Products-2024-10-01-2024-10-31.csv")

po = po.dropDuplicates()
eo = eo.dropDuplicates()

m_po = po.select(
    F.col("Order Number").alias("Order_Number").cast("int"),  # Cast to INT
    F.date_format(F.to_timestamp(F.col("Sale Date"), "M/d/yyyy h:mm:ss a"), "yyyy-MM-dd HH:mm:ss").alias("Sale_Date"),  # Format for Redshift
    F.col("Customer Class").alias("Customer_Class"),  # String type
    F.col("BillFirstName").alias("Bill_First_Name"),  # String type
    F.col("BillLastName").alias("Bill_Last_Name"),  # String type
    F.col("BillAddress").alias("Bill_Address"),  # String type
    F.col("BillCity").alias("Bill_City"),  # String type
    F.col("BillState").alias("Bill_State"),  # String type
    F.col("BillZip").alias("Bill_Zip"),  # String type
    F.col("Title").alias("Title"),  # String type
    F.col("Qty").cast("int").alias("Qty"),  # Cast to INT
    F.col("UnitPrice").cast("decimal(10, 2)").alias("Unit_Price"),  # Cast to DECIMAL 
    F.col("Revenue").cast("decimal(10, 2)").alias("Revenue")  # Cast to DECIMAL
)

m_eo = eo.select(
    F.col("Order Number").alias("Order_Number").cast("int"),  # Cast to INT
    F.date_format(F.to_timestamp(F.col("Sale Date"), "M/d/yyyy h:mm:ss a"), "yyyy-MM-dd HH:mm:ss").alias("Sale_Date"),  # Format for Redshift
    F.col("Customer Class").alias("Customer_Class"),  # String type
    F.col("BillFirstName").alias("Bill_First_Name"),  # String type
    F.col("BillLastName").alias("Bill_Last_Name"),  # String type
    F.col("BillAddress").alias("Bill_Address"),  # String type
    F.col("BillCity").alias("Bill_City"),  # String type
    F.col("BillState").alias("Bill_State"),  # String type
    F.col("BillZip").alias("Bill_Zip"),  # String type
    F.col("Title").alias("Title"),  # String type
    F.col("Qty").cast("int").alias("Qty"),  # Cast to INT
    F.col("UnitPrice").cast("decimal(10, 2)").alias("Unit_Price"),  # Cast to DECIMAL
    F.col("Revenue").cast("decimal(10, 2)").alias("Revenue")  # Cast to DECIMAL
)


In [9]:
# Loading the files to S3 after casting

s3_path_m_po = "s3://pearmundeffinghamwinery/Output/OrderPort/pc_oct"
s3_path_m_eo = "s3://pearmundeffinghamwinery/Output/OrderPort/em_oct"


m_po.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_m_po)
m_eo.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_m_eo)

In [2]:
### Customer Data

pc_tock_cust = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/Tock/pearmundcellars-24145-Tock-Guests-2024-11-06.csv")
pc_OP_cust = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/OrderPort/pearmundcellars-op-Customer-Account.11-06-2024.csv")
pc_cc_cust = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/ConstantContact/pearmundcellars_cc_contact_export_2024-11-06.csv")

em_tock_cust = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/Tock/effinghammanor-24146-Tock-Guests-2024-11-06.csv")
em_OP_cust = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/OrderPort/effinghammanor-op-Customer-Account.11-06-2024.csv")
em_cc_cust = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/ConstantContact/effinghammanor_cc_contact_export_2024-11-06.csv")


In [5]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.functions import initcap

# Convert first_name, last_name, country, city, and state to proper case; email to lowercase in pc_tock_cust
pc_tock_cust = pc_tock_cust.select(
    F.initcap(F.regexp_replace(F.col("first_name"), "[^a-zA-Z]", "")).cast(T.StringType()).alias("first_name"),
    F.initcap(F.regexp_replace(F.col("last_name"), "[^a-zA-Z]", "")).cast(T.StringType()).alias("last_name"),
    F.lower(F.regexp_replace(F.col("email"), "[\'\"]", "")).cast(T.StringType()).alias("email"),
    F.regexp_replace(F.col("phone"), "[^0-9]", "").cast(T.StringType()).alias("phone"),
    F.col("address").cast(T.StringType()).alias("address"),
    F.initcap(F.col("city")).cast(T.StringType()).alias("city"),
    F.initcap(F.col("state")).cast(T.StringType()).alias("state"),
    F.initcap(F.col("country")).cast(T.StringType()).alias("country"),
    F.col("zip_code").cast(T.StringType()).alias("zip_code")
)

# Convert first_name, last_name, country, city, and state to proper case; email to lowercase in em_tock_cust
em_tock_cust = em_tock_cust.select(
    F.initcap(F.regexp_replace(F.col("first_name"), "[^a-zA-Z]", "")).cast(T.StringType()).alias("first_name"),
    F.initcap(F.regexp_replace(F.col("last_name"), "[^a-zA-Z]", "")).cast(T.StringType()).alias("last_name"),
    F.lower(F.regexp_replace(F.col("email"), "[\'\"]", "")).cast(T.StringType()).alias("email"),
    F.regexp_replace(F.col("phone"), "[^0-9]", "").cast(T.StringType()).alias("phone"),
    F.col("address").cast(T.StringType()).alias("address"),
    F.initcap(F.col("city")).cast(T.StringType()).alias("city"),
    F.initcap(F.col("state")).cast(T.StringType()).alias("state"),
    F.initcap(F.col("country")).cast(T.StringType()).alias("country"),
    F.col("zip_code").cast(T.StringType()).alias("zip_code")
)

In [6]:
# Loading the files to S3 after casting

s3_path_pc_cust = "s3://pearmundeffinghamwinery/Output/Tock/pc_cust_oct"
s3_path_em_cust = "s3://pearmundeffinghamwinery/Output/Tock/em_cust_oct"

pc_tock_cust.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_pc_cust)
em_tock_cust.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_em_cust)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.functions import initcap

# Apply transformations to em_tock_cust using select with column renaming and casting
pc_OP_cust = pc_OP_cust.select(
    F.col("Cust #").cast(T.IntegerType()).alias("customer_number"),
    initcap(F.regexp_replace(F.col("BillFirstName"), "[^a-zA-Z]", "")).cast(T.StringType()).alias("first_name"),
    initcap(F.regexp_replace(F.col("BillLastName"), "[^a-zA-Z]", "")).cast(T.StringType()).alias("last_name"),
    F.lower(F.regexp_replace(F.col("BillEmail"), "[\'\"]", "")).cast(T.StringType()).alias("email"),
    initcap(F.col("BillState")).cast(T.StringType()).alias("state"),
    initcap(F.col("BillCity")).cast(T.StringType()).alias("city"),
    F.col("BillZip").cast(T.StringType()).alias("zip_code"),
    F.col("BillAddress").cast(T.StringType()).alias("address"),
    initcap(F.col("BillCountry")).cast(T.StringType()).alias("country"),
    F.col("Cust. Class").cast(T.StringType()).alias("customer_class")
)


# Apply transformations to em_tock_cust using select with column renaming and casting
em_OP_cust = em_OP_cust.select(
    F.col("Cust #").cast(T.IntegerType()).alias("customer_number"),
    initcap(F.regexp_replace(F.col("BillFirstName"), "[^a-zA-Z]", "")).cast(T.StringType()).alias("first_name"),
    initcap(F.regexp_replace(F.col("BillLastName"), "[^a-zA-Z]", "")).cast(T.StringType()).alias("last_name"),
    F.lower(F.regexp_replace(F.col("BillEmail"), "[\'\"]", "")).cast(T.StringType()).alias("email"),
    initcap(F.col("BillState")).cast(T.StringType()).alias("state"),
    initcap(F.col("BillCity")).cast(T.StringType()).alias("city"),
    F.col("BillZip").cast(T.StringType()).alias("zip_code"),
    F.col("BillAddress").cast(T.StringType()).alias("address"),
    initcap(F.col("BillCountry")).cast(T.StringType()).alias("country"),
    F.col("Cust. Class").cast(T.StringType()).alias("customer_class")
)


In [10]:
# Loading the files to S3 after casting

s3_path_pc_cust = "s3://pearmundeffinghamwinery/Output/OrderPort/pc_cust_oct"
s3_path_em_cust = "s3://pearmundeffinghamwinery/Output/OrderPort/em_cust_oct"


pc_OP_cust.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_pc_cust)
em_OP_cust.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_em_cust)

In [7]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# For pc_cust DataFrame
pc_cc_cust = pc_cc_cust.select(
    F.regexp_replace(F.col("Email address"), "[\'\"]", "").cast(T.StringType()).alias("Email_address"),
    F.regexp_replace(F.col("First name"), "[^a-zA-Z]", "").cast(T.StringType()).alias("First_name"),
    F.regexp_replace(F.col("Last name"), "[^a-zA-Z]", "").cast(T.StringType()).alias("Last_name"),
    F.regexp_replace(F.col("Phone - home"), "[^0-9]", "").cast(T.StringType()).alias("Phone_home"),
    F.col("Street address line 1 - Home").cast(T.StringType()).alias("Street_address_line_1_Home"),
    F.col("City - Home").cast(T.StringType()).alias("City_Home"),
    F.col("State/Province - Home").cast(T.StringType()).alias("State_Province_Home"),
    F.regexp_replace(F.col("Zip/Postal Code - Home"), "[^0-9]", "").cast(T.StringType()).alias("Zip_Postal_Code_Home"),
    F.col("Country - Home").cast(T.StringType()).alias("Country_Home")
)
 
# For em_cust DataFrame
em_cc_cust = em_cc_cust.select(
    F.regexp_replace(F.col("Email address"), "[\'\"]", "").cast(T.StringType()).alias("Email_address"),
    F.regexp_replace(F.col("First name"), "[^a-zA-Z]", "").cast(T.StringType()).alias("First_name"),
    F.regexp_replace(F.col("Last name"), "[^a-zA-Z]", "").cast(T.StringType()).alias("Last_name"),
    F.regexp_replace(F.col("Phone - home"), "[^0-9]", "").cast(T.StringType()).alias("Phone_home"),
    F.col("Street address line 1 - Home").cast(T.StringType()).alias("Street_address_line_1_Home"),
    F.col("City - Home").cast(T.StringType()).alias("City_Home"),
    F.col("State/Province - Home").cast(T.StringType()).alias("State_Province_Home"),
    F.regexp_replace(F.col("Zip/Postal Code - Home"), "[^0-9]", "").cast(T.StringType()).alias("Zip_Postal_Code_Home"),
    F.col("Country - Home").cast(T.StringType()).alias("Country_Home")
)

In [8]:
# Loading the files to S3 after casting

s3_path_pc_cust = "s3://pearmundeffinghamwinery/Output/ConstantContact/pc_cust_oct"
s3_path_em_cust = "s3://pearmundeffinghamwinery/Output/ConstantContact/em_cust_oct"


pc_cc_cust.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_pc_cust)
em_cc_cust.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_em_cust)

In [2]:
df1 = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/Output/Tock/em_cust_oct/part-00000-88618920-d268-4c95-ad10-de506fd22f2a-c000.csv")
df2 = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/Output/Tock/em_cust/part-00000-5d64b078-9634-4ad8-b5dc-19d998dbbf15-c000.csv")

# Merge the two DataFrames
merged_df = df1.union(df2)

# Remove duplicates based on the columns you want (e.g., first_name, last_name, email)
cleaned_df = merged_df.dropDuplicates()

In [3]:
s3_path_em_cust = "s3://pearmundeffinghamwinery/Output/Tock/customer_em"
cleaned_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_em_cust)

7954


In [ ]:
df1 = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/Output/Tock/em_cust_oct/part-00000-88618920-d268-4c95-ad10-de506fd22f2a-c000.csv")
df2 = spark.read.option("header", "true").csv("s3://pearmundeffinghamwinery/Output/Tock/em_cust/part-00000-5d64b078-9634-4ad8-b5dc-19d998dbbf15-c000.csv")

# Merge the two DataFrames
merged_df = df1.union(df2)

# Remove duplicates based on the columns you want (e.g., first_name, last_name, email)
cleaned_df = merged_df.dropDuplicates()

In [ ]:
s3_path_pc_cust = "s3://pearmundeffinghamwinery/Output/Tock/customer_pc"
pc_cc_cust.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_pc_cust)

In [6]:
op_em = spark.read.option("header", "true") \
                  .option("delimiter", ",") \
                  .option("quote", "\"") \
                  .option("escape", "\\") \
                  .option("multiLine", "true") \
                  .option("inferSchema", "true") \
                  .csv('s3://pearmundeffinghamwinery/OrderPort/effinghammanor-sales-master-custom.csv')

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType

# Applying transformations and parsing date correctly
op_em_df = op_em.select(
    F.col("CustomerNumber").cast(IntegerType()).alias("customer_number"),
    F.initcap(F.regexp_replace(F.col("BillFirstName"), "[^a-zA-Z]", "")).cast(StringType()).alias("first_name"),
    F.initcap(F.regexp_replace(F.col("BillLastName"), "[^a-zA-Z]", "")).cast(StringType()).alias("last_name"),
    F.lower(F.regexp_replace(F.col("BillEmail"), "[\'\"]", "")).cast(StringType()).alias("email"),
    F.col("OrderNumber").cast(IntegerType()).alias("order_number"),
    F.date_format(F.to_timestamp(F.col("SaleDate"), "MM/dd/yyyy h:mm a"), "yyyy-MM-dd HH:mm:ss").alias("sale_date"),
    F.col("OrderStatus").cast(StringType()).alias("order_status"),
    F.col("CustomerClass").cast(StringType()).alias("customer_class"),
    F.col("OrderedItems").cast(StringType()).alias("ordered_items"),
    F.regexp_replace(F.col("OrderGrandTotal"), "[$,]", "").cast(DoubleType()).alias("order_grand_total"),
    F.col("OrderBottlesOfWine").cast(DoubleType()).alias("order_bottles_of_wine"),
    F.initcap(F.regexp_replace(F.col("ProductTitle"), "[^a-zA-Z0-9 ]", "")).cast(StringType()).alias("product_title"),
    F.initcap(F.regexp_replace(F.col("ProductType"), "[^a-zA-Z ]", "")).cast(StringType()).alias("product_type"),
    F.initcap(F.col("CatalogCategory")).cast(StringType()).alias("catalog_category"),
    F.initcap(F.col("ProductGroup")).cast(StringType()).alias("product_group")
)

In [8]:
op_em_df.show()

+---------------+-------------+---------+--------------------+------------+-------------------+------------+--------------+--------------------+-----------------+---------------------+--------------------+------------+------------------+--------------------+
|customer_number|   first_name|last_name|               email|order_number|          sale_date|order_status|customer_class|       ordered_items|order_grand_total|order_bottles_of_wine|       product_title|product_type|  catalog_category|       product_group|
+---------------+-------------+---------+--------------------+------------+-------------------+------------+--------------+--------------------+-----------------+---------------------+--------------------+------------+------------------+--------------------+
|           1052|        Carla|    Moyer|  cmoyer30@gmail.com|        1008|2024-01-02 12:25:00|    Released|      Consumer|1 x 2020 Kings Ra...|           244.44|                  6.0|       2019 Meritage|        Wine|     

In [9]:
s3_path_em = "s3://pearmundeffinghamwinery/Output/OrderPort/sales_em"
op_em_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_em)

In [10]:
op_pc = spark.read.option("header", "true") \
                  .option("delimiter", ",") \
                  .option("quote", "\"") \
                  .option("escape", "\\") \
                  .option("multiLine", "true") \
                  .option("inferSchema", "true") \
                  .csv('s3://pearmundeffinghamwinery/OrderPort/pearmundcellars-sales-master-custom.csv')

In [11]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType

# Applying transformations and parsing date correctly
op_pc_df = op_pc.select(
    F.col("CustomerNumber").cast(IntegerType()).alias("customer_number"),
    F.initcap(F.regexp_replace(F.col("BillFirstName"), "[^a-zA-Z]", "")).cast(StringType()).alias("first_name"),
    F.initcap(F.regexp_replace(F.col("BillLastName"), "[^a-zA-Z]", "")).cast(StringType()).alias("last_name"),
    F.lower(F.regexp_replace(F.col("BillEmail"), "[\'\"]", "")).cast(StringType()).alias("email"),
    F.col("OrderNumber").cast(IntegerType()).alias("order_number"),
    F.date_format(F.to_timestamp(F.col("SaleDate"), "MM/dd/yyyy h:mm a"), "yyyy-MM-dd HH:mm:ss").alias("sale_date"),
    F.col("OrderStatus").cast(StringType()).alias("order_status"),
    F.col("CustomerClass").cast(StringType()).alias("customer_class"),
    F.col("OrderedItems").cast(StringType()).alias("ordered_items"),
    F.regexp_replace(F.col("OrderGrandTotal"), "[$,]", "").cast(DoubleType()).alias("order_grand_total"),
    F.col("OrderBottlesOfWine").cast(DoubleType()).alias("order_bottles_of_wine"),
    F.initcap(F.regexp_replace(F.col("ProductTitle"), "[^a-zA-Z0-9 ]", "")).cast(StringType()).alias("product_title"),
    F.initcap(F.regexp_replace(F.col("ProductType"), "[^a-zA-Z ]", "")).cast(StringType()).alias("product_type"),
    F.initcap(F.col("CatalogCategory")).cast(StringType()).alias("catalog_category"),
    F.initcap(F.col("ProductGroup")).cast(StringType()).alias("product_group")
)

In [13]:
s3_path_pc = "s3://pearmundeffinghamwinery/Output/OrderPort/sales_pc"
op_pc_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(s3_path_pc)

In [12]:
op_pc_df.show()

+---------------+----------+---------+--------------------+------------+-------------------+------------+--------------+--------------------+-----------------+---------------------+--------------------+------------+----------------+--------------------+
|customer_number|first_name|last_name|               email|order_number|          sale_date|order_status|customer_class|       ordered_items|order_grand_total|order_bottles_of_wine|       product_title|product_type|catalog_category|       product_group|
+---------------+----------+---------+--------------------+------------+-------------------+------------+--------------+--------------------+-----------------+---------------------+--------------------+------------+----------------+--------------------+
|           1062|  Pearmund|  Cellars|info@pearmundcell...|        1004|2024-01-02 10:22:00|    Released|      Consumer|1 x Small Bag Chi...|             1.58|                  0.0|     Small Bag Chips|    Physical|            null|      